# STA 141B Final Project
## Geographical Analysis of Ticketmaster Events in the United States
### Peyton Lee, Joy Liu, Katriel Layam, Leah Aviles

In [33]:
# import packages
import requests
import json
import pandas as pd
import time
import folium
import sqlite3

from datetime import datetime

To access the ticketmaster API, each person is assigned a unique API consumer key. We each created txt files that have our API keys and stored it
in a filepath string variable for easier and cleaner access to our key.

In [ ]:
# PEYTON RUN
filepath = "/Users/peyton/Desktop/TicketmasterAPIKey.txt"

In [2]:
# KATRIEL RUN
filepath = "/Users/katriellayam/Downloads/ticketmaster_project.txt"

In [ ]:
# LEAH RUN
filepath = "/Users/leahaviles/Downloads/ticket_masterAPIKey.txt"

In [ ]:
# JOY RUN
filepath = "/Users/joy/Downloads/ticketmaster api key.txt"

In [3]:
# function read_key that reads in API key txt file and returns stripped version of key
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

# use read_key function on our filepath and set it as key variable to easily access API key
key = read_key(filepath)

# check type of the key
type(key)

str

Create Dataframe of all ticketmaster events from March 2026 to February 2027

In [ ]:
# RUN ONCE
# this time we fetch 'classifications' gives if its music, sport, etc.
all_events = []

def fetch_events(start, end, label):
    page = 0
    chunk_events = []

    while page < 5:
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)

        if response.status_code == 400:
            print(f"  400 error on page {page+1}, stopping this chunk.")
            break

        response.raise_for_status()
        data = response.json()

        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            try:
                venue = e.get("_embedded", {}).get("venues", [{}])[0]
                loc = venue.get("location", {})

                # extract classifications
                classifications = e.get("classifications", [{}])
                primary = classifications[0] if classifications else {}
                segment  = primary.get("segment", {}).get("name")
                genre    = primary.get("genre", {}).get("name")
                subgenre = primary.get("subGenre", {}).get("name")

                chunk_events.append({
                    "name":     e.get("name"),
                    "date":     e.get("dates", {}).get("start", {}).get("localDate"),
                    "venue":    venue.get("name"),
                    "city":     venue.get("city", {}).get("name"),
                    "state":    venue.get("state", {}).get("stateCode"),
                    "latitude":  loc.get("latitude"),
                    "longitude": loc.get("longitude"),
                    "url":      e.get("url"),
                    "month":    start[:7],
                    "segment":  segment,
                    "genre":    genre,
                    "subgenre": subgenre,
                })
            except Exception as ex:
                print(f" Skipping event: {ex}")
                continue

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(chunk_events)} events so far")

        if page + 1 >= total_pages:
            break

        page += 1
        time.sleep(0.25)

    return chunk_events

month_chunks = [
    # March 2026
    ("2026-03-01", "2026-03-07"), ("2026-03-08", "2026-03-15"),
    ("2026-03-16", "2026-03-23"), ("2026-03-24", "2026-03-31"),
    # April
    ("2026-04-01", "2026-04-07"), ("2026-04-08", "2026-04-15"),
    ("2026-04-16", "2026-04-23"), ("2026-04-24", "2026-04-30"),
    # May
    ("2026-05-01", "2026-05-07"), ("2026-05-08", "2026-05-15"),
    ("2026-05-16", "2026-05-23"), ("2026-05-24", "2026-05-31"),
    # June
    ("2026-06-01", "2026-06-07"), ("2026-06-08", "2026-06-15"),
    ("2026-06-16", "2026-06-23"), ("2026-06-24", "2026-06-30"),
    # July
    ("2026-07-01", "2026-07-07"), ("2026-07-08", "2026-07-15"),
    ("2026-07-16", "2026-07-23"), ("2026-07-24", "2026-07-31"),
    # August
    ("2026-08-01", "2026-08-07"), ("2026-08-08", "2026-08-15"),
    ("2026-08-16", "2026-08-23"), ("2026-08-24", "2026-08-31"),
    # September
    ("2026-09-01", "2026-09-07"), ("2026-09-08", "2026-09-15"),
    ("2026-09-16", "2026-09-23"), ("2026-09-24", "2026-09-30"),
    # October
    ("2026-10-01", "2026-10-07"), ("2026-10-08", "2026-10-15"),
    ("2026-10-16", "2026-10-23"), ("2026-10-24", "2026-10-31"),
    # November
    ("2026-11-01", "2026-11-07"), ("2026-11-08", "2026-11-15"),
    ("2026-11-16", "2026-11-23"), ("2026-11-24", "2026-11-30"),
    # December
    ("2026-12-01", "2026-12-07"), ("2026-12-08", "2026-12-15"),
    ("2026-12-16", "2026-12-23"), ("2026-12-24", "2026-12-31"),
    # January 2027
    ("2027-01-01", "2027-01-07"), ("2027-01-08", "2027-01-15"),
    ("2027-01-16", "2027-01-23"), ("2027-01-24", "2027-01-31"),
    # February 2027
    ("2027-02-01", "2027-02-07"), ("2027-02-08", "2027-02-15"),
    ("2027-02-16", "2027-02-23"), ("2027-02-24", "2027-02-28"),
]

for start, end in month_chunks:
    label = start[:7]
    events = fetch_events(start, end, label)
    all_events.extend(events)
    print(f"Running total: {len(all_events)} events \n")
    time.sleep(0.25)

events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"])
events_df["latitude"]  = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)
events_df = events_df.drop_duplicates(subset=["name", "date", "venue"])

print(f"\nTotal events: {len(events_df)}")
print(events_df[["segment", "genre", "subgenre"]].value_counts().head(20))

output_file = "classified_ticketmaster_events.csv"
events_df.to_csv(output_file, index=False)

print(f"Saved {len(events_df)} events to {output_file}")

In [3]:
# data cleaning!! so awesome
# load from csv
events_df = pd.read_csv("classified_ticketmaster_events.csv")

# look at the data (EXPLORE)
events_df.head()
print(events_df.shape)
print(events_df.head())
print(events_df.info())
print(events_df.isna().sum())


(33978, 12)
                                           name        date  \
0                  Museum of Chinese in America  2026-03-04   
1                     2025 Bulls Bucks Vouchers  2026-02-20   
2                           A-List ID Card 2025  2026-02-05   
3                 EXP ROSEMONT - Flexible Entry  2026-02-02   
4  Community Concerts of the Grand Valley 25-26  2026-01-04   

                          venue            city state   latitude   longitude  \
0  Museum of Chinese in America        New York    NY  40.719533  -73.999029   
1    Durham Bulls Athletic Park          Durham    NC  35.991872  -78.904015   
2                Atlanta Braves             NaN   NaN  33.890304  -84.468127   
3                EXP - Rosemont        Rosemont    IL  41.975466  -87.872811   
4                Avalon Theatre  Grand Junction    CO  39.067000 -108.561516   

                                                 url    month        segment  \
0  https://www.universe.com/events/museum-of-chi

In [14]:
# explore rows more closely
# inspect rows with missing values
events_df[events_df["state"].isna()].head(20)
events_df[events_df["city"].isna()].head(20)
events_df[events_df["segment"].isna()].head(20)
events_df[events_df["genre"].isna()].head(20)
events_df[events_df["subgenre"].isna()].head(20)

# checking for unique values
sorted(events_df["state"].dropna().unique())

events_df["segment"].value_counts()

events_df["genre"].value_counts().head(20)

events_df["subgenre"].value_counts().head(20)

# explore locations
events_df["state"].value_counts().head(20)

events_df["city"].value_counts().head(20)

# explore time frame
events_df["month"].value_counts().sort_index()


month
2026-03    2531
2026-04    3482
2026-05    3467
2026-06    3315
2026-07    3269
2026-08    3225
2026-09    3337
2026-10    2948
2026-11    3234
2026-12    2630
2027-01    1559
2027-02     981
Name: count, dtype: int64

In [29]:
# START CLEANING
# some events have abbreviations, some have full names like "Texas" and "TX" are both options
# we coerce them to be the same
state_name_to_abbr = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

def normalize_state(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    if len(val) == 2:
        return val.upper()
    return state_name_to_abbr.get(val, None)

events_clean = events_df.copy()

# strip text columns
events_clean["name"] = events_clean["name"].str.strip()
events_clean["venue"] = events_clean["venue"].str.strip()
# only strip city if it exists
events_clean["city"] = events_clean["city"].where(
    events_clean["city"].isna(),
    events_clean["city"].str.strip().str.title()
)

# apply function to normalize states
events_clean["state"] = events_clean["state"].apply(normalize_state)

# fill missing segment, genre, subgenre categories
events_clean["segment"] = events_clean["segment"].fillna("Unknown")
events_clean["genre"] = events_clean["genre"].fillna("Unknown")
events_clean["subgenre"] = events_clean["subgenre"].fillna("Unknown")

# remove ticket/products/non-events where possible, trying to preserve real events (if they have null) as much as possible
events_clean = events_clean[
    ~events_clean["name"].str.lower().str.contains(
        "voucher|package|flex|pass|seating|deposit|suite|discount|card",
        na=False
    )
]

# helper columns for dropping duplicates
events_clean["name_key"] = events_clean["name"].str.lower()
events_clean["venue_key"] = events_clean["venue"].str.lower()

# dropping duplicates
events_clean = events_clean.drop_duplicates(
    subset=["name_key", "date", "venue_key"]
).drop(columns=["name_key", "venue_key"])


# fill missing city/state only if needed for analysis (MAYBE DELETE?)
# events_clean["city"] = events_clean["city"].fillna("unknown")
# events_clean["state"] = events_clean["state"].fillna("unknown")

# reset index
events_clean = events_clean.reset_index(drop=True)


In [30]:
# checking cleaned data
# NOTE: CITY AND STATE STILL HAVE UNKNOWNS, BUT THEY ALSO HAVE LAT AND LONG COORDINATES, so they exist and are lowkey valid??

print("Original rows:", len(events_df))
print("Cleaned rows:", len(events_clean))
print("Missing values after cleaning:")
print(events_clean.isna().sum())

print("unique state values after cleaning:")
print(sorted(events_clean["state"].dropna().unique()))

events_clean.head(20)

Original rows: 33978
Cleaned rows: 31368
Missing values after cleaning:
name            0
date            0
venue           0
city          849
state         974
latitude        0
longitude       0
url          7689
month           0
segment         0
genre           0
subgenre        0
dtype: int64
unique state values after cleaning:
['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']


,name,date,venue,city,state,latitude,longitude,url,month,segment,genre,subgenre
0,Museum of Chinese in America,2026-03-04,Museum of Chinese in America,New York,NY,40.719533,-73.999029,https://www.universe.com/events/museum-of-chin...,2026-03,Miscellaneous,Community/Civic,Community/Civic
1,Community Concerts of the Grand Valley 25-26,2026-01-04,Avalon Theatre,Grand Junction,CO,39.067000,-108.561516,https://www.ticketmaster.com/community-concert...,2026-03,Music,Other,Other
2,BLUE FRIDAY SPECIAL! Entry to All Shows at the...,2026-01-01,Blue Moon Saloon,Lafayette,LA,30.220444,-92.016281,https://www.ticketweb.com/event/blue-friday-sp...,2026-03,Music,World,World
3,I'm On the List! + 1 (Entry to all shows at th...,2026-01-01,Blue Moon Saloon,Lafayette,LA,30.220444,-92.016281,https://www.ticketweb.com/event/im-on-the-list...,2026-03,Music,World,World
4,GIVEX,2025-09-04,KeyBank Center,Buffalo,NY,42.874941,-78.876224,NaN,2026-03,Unknown,Unknown,Unknown
5,Bay Breakers Season Ticket,2025-03-08,Grape Bowl,Lodi,CA,38.141210,-121.267186,https://www.universe.com/events/bay-breakers-s...,2026-03,Sports,Rugby,Rugby League
6,Ride and Dine with Us! Thriller Speedboat Tours,2026-03-06,Hard Rock Cafe Miami,Miami,FL,25.778435,-80.188001,https://www.ticketweb.com/event/ride-and-dine-...,2026-03,Miscellaneous,Undefined,Undefined
7,Ride and Dine with Us! Thriller Speedboat Tours,2026-03-03,Hard Rock Cafe Miami,Miami,FL,25.778435,-80.188001,https://www.ticketweb.com/event/ride-and-dine-...,2026-03,Miscellaneous,Undefined,Undefined
8,Ride and Dine with Us! Thriller Speedboat Tours,2026-03-04,Hard Rock Cafe Miami,Miami,FL,25.778435,-80.188001,https://www.ticketweb.com/event/ride-and-dine-...,2026-03,Miscellaneous,Undefined,Undefined
9,Ride and Dine with Us! Thriller Speedboat Tours,2026-03-05,Hard Rock Cafe Miami,Miami,FL,25.778435,-80.188001,https://www.ticketweb.com/event/ride-and-dine-...,2026-03,Miscellaneous,Undefined,Undefined


In [31]:
# EXPORT CLEANED DATASET TO CSV
output_file = "cleaned_ticketmaster_events.csv"
events_clean.to_csv(output_file, index=False)

In [35]:
# load from csv
cleaned_events = pd.read_csv("cleaned_ticketmaster_events.csv")

# create in memory db and load dfs
conn = sqlite3.connect(":memory:")
cleaned_events.to_sql("events", conn, index=False, if_exists="replace")
df.to_sql("venues",        conn, index=False, if_exists="replace")

# test
pd.read_sql("""
    SELECT segment, COUNT(*) as count
    FROM events
    GROUP BY segment
    ORDER BY count DESC
""", conn)

NameError: name 'df' is not defined

In [34]:
# only include events in mainland US
cleaned_events = pd.read_sql("""
    SELECT *
    FROM events
    WHERE latitude BETWEEN 24.5 AND 49.5
    AND longitude BETWEEN -125.0 AND -66.5
""", conn)

print(f"Events before: {len(cleaned_events)}")

# update the sql table
cleaned_events.to_sql("events", conn, index=False, if_exists="replace")

print(f"Events after: {len(cleaned_events)}")

cleaned_events

NameError: name 'conn' is not defined

In [36]:
# only include events in US
# categorize events by types of color

m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

# color by segment
segment_colors = {
    "Music": "blue",
    "Sports": "red",
    "Arts & Theatre": "green",
    "Miscellaneous": "gray",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in cleaned_events.iterrows():
    month = row["month"]
    if month not in groups:
        continue

    color = segment_colors.get(row["segment"], "gray")

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}<br>"
        f"Segment: {row['segment']}<br>"
        f"Genre: {row['genre']}"
    )

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [37]:
# make chorolopleth map colored by most popular genre in each state
import plotly.express as px

# find most popular genre per state, excluding vague ones
top_genre_by_state = (
    cleaned_events[~cleaned_events["genre"].isin(["Undefined", "Miscellaneous", "Other"])]
    .groupby(["state", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .drop_duplicates(subset="state")  # keep top genre per state
)

fig = px.choropleth(
    top_genre_by_state,
    locations="state",
    locationmode="USA-states",
    color="genre",
    scope="usa",
    hover_name="state",
    hover_data={"count": True, "genre": True},
    title="Most Popular Event Genre by State (2026)",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.update_layout(
    legend_title="Top Genre",
    title_font_size=16,
)

fig.show()

In [38]:
# can toggle event locations per month
m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in cleaned_events.iterrows():
    month = row["month"]
    if month not in groups:
        continue
    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="darkblue",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [39]:
# plot categories
import matplotlib.pyplot as plt

genre_counts = (
    cleaned_events[~cleaned_events["genre"].isin(["Undefined", "Miscellaneous", "Other"])]
    .groupby(["segment", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# color by segment
segment_colors = {
    "Music": "steelblue",
    "Sports": "firebrick",
    "Arts & Theatre": "mediumseagreen",
    "Miscellaneous": "gray",
}
colors = genre_counts["segment"].map(segment_colors).fillna("gray")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    genre_counts["genre"] + " (" + genre_counts["segment"] + ")",
    genre_counts["count"],
    color=colors
)

ax.set_xlabel("Number of Events")
ax.set_title("Top 20 Event Genres on Ticketmaster (2026)", fontsize=14)
ax.invert_yaxis()

# legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in segment_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

In [40]:
events_per_state = (
    pd.read_sql("""
        SELECT state, COUNT(*) as event_count
        FROM events
        WHERE state IS NOT NULL
        GROUP BY state
        ORDER BY event_count DESC
    """, conn)
)

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(events_per_state["state"], events_per_state["event_count"], color="steelblue")
ax.set_xlabel("State")
ax.set_ylabel("Number of Events")
ax.set_title("Number of Events per State (2026)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Below I am going to analyze the proximity of events to large US cities.

In [41]:
# intiialize coords of large US cities
major_cities = {
    "New York":       (40.7128, -74.0060),
    "Los Angeles":    (34.0522, -118.2437),
    "Chicago":        (41.8781, -87.6298),
    "Houston":        (29.7604, -95.3698),
    "Phoenix":        (33.4484, -112.0740),
    "Philadelphia":   (39.9526, -75.1652),
    "San Antonio":    (29.4241, -98.4936),
    "San Diego":      (32.7157, -117.1611),
    "Dallas":         (32.7767, -96.7970),
    "Jacksonville":   (30.3322, -81.6557),
    "Fort Worth":     (32.7555, -97.3308),
    "San Jose":       (37.3382, -121.8863),
    "Austin":         (30.2672, -97.7431),
    "Charlotte":      (35.2271, -80.8431),
    "Columbus":       (39.9612, -82.9988),
    "Indianapolis":   (39.7684, -86.1581),
    "San Francisco":  (37.7749, -122.4194),
    "Seattle":        (47.6062, -122.3321),
    "Denver":         (39.7392, -104.9903),
    "Oklahoma City":  (35.4676, -97.5164),
    "Nashville":      (36.1627, -86.7816),
    "Washington DC":  (38.9072, -77.0369),
    "El Paso":        (31.7619, -106.4850),
    "Las Vegas":      (36.1699, -115.1398),
    "Boston":         (42.3601, -71.0589),
}

cities_df = pd.DataFrame(
    [(city, lat, lon) for city, (lat, lon) in major_cities.items()],
    columns=["city_name", "city_lat", "city_lon"]
)

In [42]:
# calculate the distance to the nearest US city
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def nearest_city(lat, lon):
    best_city, best_dist = None, float("inf")
    for _, row in cities_df.iterrows():
        d = haversine(lat, lon, row["city_lat"], row["city_lon"])
        if d < best_dist:
            best_dist = d
            best_city = row["city_name"]
    return best_city, round(best_dist, 1)

# apply to cleaned_events
cleaned_events[["nearest_city", "dist_to_city_miles"]] = cleaned_events.apply(
    lambda row: nearest_city(row["latitude"], row["longitude"]),
    axis=1, result_type="expand"
)

cleaned_events.to_sql("events", conn, index=False, if_exists="replace")

In [43]:
# distribution of distances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram of distances
axes[0].hist(cleaned_events["dist_to_city_miles"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Distance to Nearest Major City (miles)")
axes[0].set_ylabel("Number of Events")
axes[0].set_title("How Far Are Events from Major Cities?")

# % of events within thresholds
thresholds = [10, 25, 50, 100, 200]
pcts = [
    (cleaned_events["dist_to_city_miles"] <= t).mean() * 100
    for t in thresholds
]
axes[1].bar([f"≤{t} mi" for t in thresholds], pcts, color="mediumseagreen")
axes[1].set_ylabel("% of Events")
axes[1].set_title("Cumulative % of Events Within Distance of a Major City")
axes[1].set_ylim(0, 100)
for i, v in enumerate(pcts):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [44]:
# events per nearest city
pd.read_sql("""
    SELECT nearest_city, COUNT(*) as event_count,
           ROUND(AVG(dist_to_city_miles), 1) as avg_dist
    FROM events
    GROUP BY nearest_city
    ORDER BY event_count DESC
""", conn)

Ok the analysis for the proximity to major cities is between these boxes.

now we will look at prices

In [45]:
# fetch prices
all_events_price = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while page < 5:  # cap at 5 pages per month to stay within rate limits
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()
        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})

            # classifications
            classifications = e.get("classifications", [{}])
            primary = classifications[0] if classifications else {}
            segment  = primary.get("segment", {}).get("name")
            genre    = primary.get("genre", {}).get("name")

            # price ranges
            price_ranges = e.get("priceRanges", [])
            price_min = price_ranges[0].get("min") if price_ranges else None
            price_max = price_ranges[0].get("max") if price_ranges else None

            all_events_price.append({
                "name":      e.get("name"),
                "date":      e.get("dates", {}).get("start", {}).get("localDate"),
                "venue":     venue.get("name"),
                "city":      venue.get("city", {}).get("name"),
                "state":     venue.get("state", {}).get("stateCode"),
                "latitude":  loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "month":     start[:7],
                "segment":   segment,
                "genre":     genre,
                "price_min": price_min,
                "price_max": price_max,
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events_price)} events so far")
        if page + 1 >= total_pages:
            break
        page += 1
        time.sleep(0.25)

price_df = pd.DataFrame(all_events_price)

# clean up
price_df["state"] = price_df["state"].apply(normalize_state)
price_df = price_df.dropna(subset=["latitude", "longitude", "price_min", "price_max"])
price_df["latitude"]  = price_df["latitude"].astype(float)
price_df["longitude"] = price_df["longitude"].astype(float)

# filter to continental US
price_df = price_df[
    price_df["latitude"].between(24.5, 49.5) &
    price_df["longitude"].between(-125.0, -66.5)
]

# load into sqlite
price_df.to_sql("events_price", conn, index=False, if_exists="replace")

print(f"\nEvents with price data: {len(price_df)}")
print(f"Events WITHOUT price data: {len(pd.DataFrame(all_events_price)) - len(price_df)}")

In [46]:
# avg price by genre
pd.read_sql("""
    SELECT genre,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    WHERE genre NOT IN ('Undefined', 'Miscellaneous', 'Other')
    GROUP BY genre
    ORDER BY avg_min DESC
    LIMIT 15
""", conn)

In [ ]:
# avg price by state
pd.read_sql("""
    SELECT state,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    GROUP BY state
    ORDER BY avg_min DESC
""", conn)

In [49]:
# avg price by segment
pd.read_sql("""
    SELECT segment,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    GROUP BY segment
    ORDER BY avg_min DESC
""", conn)

there isn't much price information, i don't know if we should use it

In [48]:
from folium.plugins import HeatMap

m = folium.Map(location=[38, -95], zoom_start=4, tiles="CartoDB dark_matter")

heat_data = cleaned_events[["latitude", "longitude", "dist_to_city_miles"]].copy()
heat_data["weight"] = 1 / (heat_data["dist_to_city_miles"] + 1)

HeatMap(
    data=heat_data[["latitude", "longitude", "weight"]].values.tolist(),
    radius=12,
    blur=15,
    max_zoom=6,
    min_opacity=0.3,

).add_to(m)

# major city markers
for city, (lat, lon) in major_cities.items():
    folium.CircleMarker(
        location=[lat, lon],
        radius=2,
        color="white",
        fill=True,
        fill_color="white",
        fill_opacity=0.9,
        popup=folium.Popup(f"<b>{city}</b>", max_width=150),
    ).add_to(m)

m.save("event_proximity_heatmap.html")
m